# Logistic regression

Linear regression predicts a number. Classification predicts a category, and the cleanest first model is still a straight line, bent through a squashing function so its output reads as a probability. This notebook builds that model from the likelihood up, trains it by gradient descent, then breaks it on purpose.

Order: intuition, maths, cost function, from-scratch version, library version, tuning, what breaks.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from common.datasets import load_iris_flowers, make_credit_data
from common.plotting import plot_decision_boundary

SEED = 42
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=3, suppress=True)
plt.rcParams["figure.figsize"] = (6, 3.5)

## 1. Intuition

Say we want to tell two iris species apart from their petals. A straight line in the petal plane splits the plane in two. Points far on one side are almost certainly species A, points far on the other side species B, and points near the line are a coin flip.

Logistic regression turns that picture into a number. It computes a score $z=\mathbf w^\top\mathbf x+b$ that is large and positive deep inside one region, large and negative deep inside the other, and zero on the line. Then it passes $z$ through the sigmoid, which maps any score into $(0,1)$ and reads as a probability. The weights set the direction of the line and the sharpness of the transition.

It is a classifier despite the name. "Regression" is there because it regresses the log-odds, as the next section shows.

## 2. Maths

**Sigmoid.** $\sigma(z)=\dfrac{1}{1+e^{-z}}$. It gives 0.5 at $z=0$, tends to 1 for large $z$ and to 0 for very negative $z$, and has the tidy derivative $\sigma'(z)=\sigma(z)(1-\sigma(z))$.

**Why this shape.** Take the odds $p/(1-p)$ and then the log. The log-odds $\log\frac{p}{1-p}$ can be any real number, so we let it be linear: $\log\frac{p}{1-p}=z$. Solving for $p$ gives the sigmoid. Each unit of $x_j$ adds $w_j$ to the log-odds, which multiplies the odds by $e^{w_j}$. That is how to read a coefficient.

**Cost function from maximum likelihood.** Each label is a coin flip with probability $p_i=\sigma(z_i)$, a Bernoulli variable (see `02_probability/03`). The probability of one label is $p_i^{y_i}(1-p_i)^{1-y_i}$. For independent rows the likelihood is the product, and the log turns it into a sum:

$$\ell(\mathbf w)=\sum_i y_i\log p_i+(1-y_i)\log(1-p_i)$$

Maximising $\ell$ is the same as minimising its negative average, the **log-loss** (binary cross-entropy):

$$J(\mathbf w)=-\frac1n\sum_i\bigl[y_i\log p_i+(1-y_i)\log(1-p_i)\bigr]$$

Why not MSE on probabilities? Log-loss punishes a confident wrong answer without limit: predicting $p=0.01$ for a true positive costs $-\log 0.01=4.6$, and $p=10^{-6}$ costs 13.8. And with a sigmoid, MSE makes the cost surface non-convex, whereas log-loss is convex, so gradient descent has one minimum to find.

**Gradient.** The sigmoid derivative cancels the awkward terms and leaves the same shape as linear regression:

$$\nabla_{\mathbf w}J=\frac1n X^\top(\mathbf p-\mathbf y),\qquad \frac{\partial J}{\partial b}=\frac1n\sum_i(p_i-y_i)$$

There is no closed-form solution, so we iterate, exactly as in `12_gradient_descent_variants`.

**Regularisation.** Add $\frac{1}{2C}\lVert\mathbf w\rVert^2$ for L2 (the sklearn default). Note that $C$ is the inverse of the strength: small $C$ means a strong penalty. As in notebook 13 this is a Gaussian prior on the weights.

**Multiclass.** With $K$ classes use one score per class and the softmax, $p_k=e^{z_k}/\sum_j e^{z_j}$. The loss becomes the categorical cross-entropy $-\log p_{\text{true class}}$. With $K=2$ it reduces to the sigmoid case.

## 3. From scratch

Data first: versicolor against virginica, using only petal length and width so we can draw the result. These two species overlap a little, which is what makes it a fair test. I standardise the features, because gradient descent is slow on mismatched scales (notebook 10).

In [ ]:
X_all, y_all = load_iris_flowers()
mask = y_all.isin([1, 2])
X2 = X_all.loc[mask, ["petal_length", "petal_width"]].to_numpy()
y2 = (y_all[mask] == 2).astype(int).to_numpy()  # 1 = virginica
X2_tr, X2_te, y2_tr, y2_te = train_test_split(X2, y2, test_size=0.3, random_state=SEED, stratify=y2)
scaler = StandardScaler().fit(X2_tr)
Xs_tr, Xs_te = scaler.transform(X2_tr), scaler.transform(X2_te)
print("train rows:", len(y2_tr), " virginica share:", y2_tr.mean())

In [ ]:
def sigmoid(z: np.ndarray) -> np.ndarray:
    return 1 / (1 + np.exp(-z))


def log_loss_scratch(y: np.ndarray, p: np.ndarray, eps: float = 1e-12) -> float:
    p = np.clip(p, eps, 1 - eps)  # avoid log(0)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))


def fit_logistic(
    X: np.ndarray, y: np.ndarray, lr: float = 0.5, steps: int = 500, l2: float = 0.0
) -> tuple[np.ndarray, float, list[float]]:
    w, b, history = np.zeros(X.shape[1]), 0.0, []
    for _ in range(steps):
        p = sigmoid(X @ w + b)
        history.append(log_loss_scratch(y, p))
        w -= lr * (X.T @ (p - y) / len(y) + l2 * w)
        b -= lr * np.mean(p - y)
    return w, b, history


w, b, history = fit_logistic(Xs_tr, y2_tr, steps=2000, l2=1 / len(y2_tr))
plt.plot(history)
plt.xlabel("step")
plt.ylabel("log-loss")
plt.show()
print("weights:", w, "bias:", round(b, 3), " final loss:", round(history[-1], 4))

The loss falls smoothly and flattens, as a convex cost should. Because the features are standardised, the two weights are comparable: the larger one is the feature that moves the log-odds most per standard deviation.

Now check it against sklearn. Its penalised cost is $C\sum_i\text{loss}_i+\frac12\lVert\mathbf w\rVert^2$. Dividing by $Cn$ gives my average loss plus $\frac{1}{2Cn}\lVert\mathbf w\rVert^2$, so with `C=1` the penalty strength in my gradient is `1 / n`.

In [ ]:
ref = LogisticRegression(C=1.0, max_iter=1000).fit(Xs_tr, y2_tr)
print("scratch:", w, b)
print("sklearn:", ref.coef_[0], ref.intercept_[0])
p_scratch = sigmoid(Xs_te @ w + b)
print(
    "test accuracy:",
    accuracy_score(y2_te, p_scratch > 0.5),
    accuracy_score(y2_te, ref.predict(Xs_te)),
)

## 4. The library version and the decision boundary

The boundary is where $z=0$, which is a line: $w_1x_1+w_2x_2+b=0$. Shading what the model predicts shows it.

In [ ]:
clf = LogisticRegression().fit(X2_tr, y2_tr)
fig, ax = plt.subplots(figsize=(5.5, 4))
plot_decision_boundary(clf, X2_tr, y2_tr, ax, labels=("petal length (cm)", "petal width (cm)"))
plt.show()
print("train accuracy:", clf.score(X2_tr, y2_tr), " test accuracy:", clf.score(X2_te, y2_te))
print("odds multiplier per extra cm of petal width:", np.exp(clf.coef_[0][1]).round(1))

The odds multiplier is the reading of a coefficient from section 2. Each extra centimetre of petal width multiplies the odds of virginica by that factor, holding petal length fixed.

The model gives probabilities, not just labels. Use `predict_proba`, and treat 0.5 as a default threshold you can move (notebook 25).

In [ ]:
samples = np.array([[4.5, 1.4], [5.0, 1.8], [5.8, 2.1]])
print(
    pd.DataFrame(
        clf.predict_proba(samples),
        columns=["versicolor", "virginica"],
        index=["small", "borderline", "large"],
    )
)

## 5. Regularisation and the choice of C

A small `C` means a strong penalty and smaller weights, which makes probabilities less extreme. A large `C` lets the weights grow. On two features, with 70 training rows, there is little overfitting to fix, so look at what changes: the size of the weights and the confidence.

In [ ]:
rows = []
for C in (0.001, 0.01, 0.1, 1, 10, 1000):
    m = LogisticRegression(C=C, max_iter=2000).fit(Xs_tr, y2_tr)
    rows.append(
        {
            "C": C,
            "|w| max": np.abs(m.coef_).max(),
            "train log-loss": log_loss(y2_tr, m.predict_proba(Xs_tr)),
            "test log-loss": log_loss(y2_te, m.predict_proba(Xs_te)),
            "test acc": m.score(Xs_te, y2_te),
        }
    )
pd.DataFrame(rows).set_index("C").round(3)

Very strong penalties shrink the weights towards zero and the probabilities towards 0.5, so log-loss is poor even where accuracy holds up. Test log-loss is best around $C=10$ and gets slightly worse after that, as the weights grow to fit the training rows more tightly. Tune `C` by cross-validation on log-loss. L1 (`penalty="l1"`, `solver="liblinear"` or `"saga"`) zeroes weights as Lasso did, which is useful with many features.

In [ ]:
from sklearn.model_selection import GridSearchCV

search = GridSearchCV(
    LogisticRegression(max_iter=2000),
    {"C": np.logspace(-3, 3, 13)},
    cv=5,
    scoring="neg_log_loss",
).fit(Xs_tr, y2_tr)
print(
    "best C:",
    round(float(search.best_params_["C"]), 3),
    " test acc:",
    accuracy_score(y2_te, search.predict(Xs_te)),
)

## 6. More than two classes: softmax

For all three species, sklearn fits one score per class and applies the softmax automatically. I use all four measurements here and check the probabilities sum to one.

In [ ]:
Xf_tr, Xf_te, yf_tr, yf_te = train_test_split(
    X_all, y_all, test_size=0.3, random_state=SEED, stratify=y_all
)
multi = LogisticRegression(max_iter=2000).fit(Xf_tr, yf_tr)
proba = multi.predict_proba(Xf_te)
print("test accuracy:", multi.score(Xf_te, yf_te))
print("rows sum to one:", np.allclose(proba.sum(axis=1), 1))
z = Xf_te.to_numpy() @ multi.coef_.T + multi.intercept_
soft = np.exp(z) / np.exp(z).sum(axis=1, keepdims=True)
print("softmax by hand matches predict_proba:", np.allclose(soft, proba))

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4))
two = ["petal_length", "petal_width"]
m3 = LogisticRegression(max_iter=2000).fit(X_all[two].to_numpy(), y_all)
plot_decision_boundary(
    m3,
    X_all[two].to_numpy(),
    y_all.to_numpy(),
    ax,
    labels=("petal length (cm)", "petal width (cm)"),
)
plt.show()

Three regions, each meeting the next along a straight line. Linear models only ever draw straight boundaries. Notebooks 21 to 24 handle curved ones.

## 7. What breaks

**Separable data.** Setosa is perfectly separable from the other two species by petal size. If a line splits the classes with no errors, the likelihood keeps rising as the weights grow, because larger weights push every probability closer to 0 or 1. There is no finite maximum. Watch the weights as the penalty is relaxed.

In [ ]:
Xsep = X_all[["petal_length", "petal_width"]].to_numpy()
ysep = (y_all == 0).astype(int).to_numpy()
for C in (1, 100, 1e4, 1e6):
    m = LogisticRegression(C=C, max_iter=100000).fit(Xsep, ysep)
    print(
        f"C={C:>9.0e}  weight norm {np.linalg.norm(m.coef_):8.2f}  train log-loss {log_loss(ysep, m.predict_proba(Xsep)):.6f}"
    )

The weights grow with `C`, and the loss keeps shrinking towards zero. Perfect separation looks like a success and gives wildly overconfident probabilities. The regulariser is what keeps the answer finite, so leave it on by default.

**Class imbalance.** Take the synthetic credit data, with about 4% defaults. A model that always predicts "no default" scores 96% accuracy and is useless. Logistic regression trained normally pushes its probabilities down towards the base rate, so almost nothing crosses 0.5.

In [ ]:
Xc, yc = make_credit_data(seed=SEED)
Xc = Xc.fillna(Xc.median())
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.3, random_state=SEED, stratify=yc)
sc = StandardScaler().fit(Xc_tr)
for name, kw in [("default", {}), ("class_weight=balanced", {"class_weight": "balanced"})]:
    m = LogisticRegression(max_iter=1000, **kw).fit(sc.transform(Xc_tr), yc_tr)
    pred = m.predict(sc.transform(Xc_te))
    caught = pred[yc_te == 1].mean()
    print(
        f"{name:>22}: accuracy {accuracy_score(yc_te, pred):.3f}  defaults caught {caught:.2f}  flagged {pred.mean():.3f}"
    )
print("always predict no default:", round(1 - yc_te.mean(), 3))

The default model matches the do-nothing accuracy (96.2%) and catches 1% of the defaults, because it flags only 0.1% of applicants. `class_weight="balanced"` reweights the loss so each class counts equally. It catches 66% of the defaults but flags 30% of applicants, and accuracy drops to 71%. Neither is "right". The model is fine, and the 0.5 threshold is the wrong tool. Notebook 25 picks thresholds from costs instead.

**Other limits.** The boundary is linear in the features you give it. Add interaction or polynomial terms (notebook 11) for curves. Collinear features make individual weights unstable, as in notebook 10, and the penalty helps there too.

Logistic regression is the baseline every other classifier here has to beat. Next, a model with no weights at all.